# 02 -- model training

trains a small neural network on the board state data generated by
notebook 01. the model learns to predict how much score remains
from any given board position.

architecture: 16 -> dense 128 relu -> dense 64 relu -> 1 linear.

exports to tensorflow.js format (model.json + weights.bin).

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f'tensorflow version: {tf.__version__}')

tensorflow version: 2.20.0


## load and prepare data

In [2]:
import os
data_path = 'training_data.npz'
if not os.path.exists(data_path):
    data_path = '/kaggle/input/datasets/maheshkakde165/2048-training-dataset/training_data.npz'
data = np.load(data_path)
boards = data['boards']
scores = data['scores']

print(f'boards: {boards.shape}')
print(f'scores: {scores.shape}')
print(f'score range: {scores.min():.0f} - {scores.max():.0f}')

# normalize scores to [0, 1] range for training stability
score_max = scores.max()
scores_norm = scores / score_max

# shuffle and split: 90% train, 10% validation
n = len(boards)
indices = np.random.permutation(n)
split = int(n * 0.9)

x_train = boards[indices[:split]]
y_train = scores_norm[indices[:split]]
x_val = boards[indices[split:]]
y_val = scores_norm[indices[split:]]

print(f'\ntrain: {x_train.shape[0]} samples')
print(f'val:   {x_val.shape[0]} samples')

boards: (548337, 16)
scores: (548337,)
score range: 0 - 62144

train: 493503 samples
val:   54834 samples


## build model

In [3]:
model = keras.Sequential([
    layers.Input(shape=(16,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='linear')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

model.summary()

I0000 00:00:1791833811.803141      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1791833811.806624      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         2,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,497 (41.00 KB)

 Trainable params: 10,497 (41.00 KB)

 Non-trainable params: 0 (0.00 B)

## train

In [4]:
history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=50,
    batch_size=256,
    verbose=1
)

Epoch 1/50
  77/1928 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.0471 - mae: 0.1682  

I0000 00:00:1791833816.131759     142 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1928/1928 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 0.0261 - mae: 0.1261 - val_loss: 0.0253 - val_mae: 0.1240
Epoch 2/50
1928/1928 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.0250 - mae: 0.1234 - val_loss: 0.0251 - val_mae: 0.1239
Epoch 3/50
1928/1928 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.0248 - mae: 0.1226 - val_loss: 0.0250 - val_mae: 0.1228
Epoch 4/50
1928/1928 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.0246 - mae: 0.1219 - val_loss: 0.0249 - val_mae: 0.1199
Epoch 5/50
1928/1928 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.0244 - mae: 0.1213 - val_loss: 0.0246 - val_mae: 0.1221
Epoch 6/50
1928/1928 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.0242 - mae: 0.1208 - val_loss: 0.0244 - val_mae: 0.1222
Epoch 7/50
1928/1928 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.0241 - mae: 0.1204 - val_loss: 0.0244 - val_mae: 0.1201
Epoch 8/50
1928/1928 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.0240 - mae: 0.1200 - val_loss: 0.0242 - val_mae: 0.1213
Epoch 9/50
1928/1928 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/st

## evaluate

In [5]:
val_loss, val_mae = model.evaluate(x_val, y_val, verbose=0)

# convert mae back to actual score units
val_mae_score = val_mae * score_max

print(f'validation loss (mse): {val_loss:.6f}')
print(f'validation mae (normalized): {val_mae:.4f}')
print(f'validation mae (score units): {val_mae_score:.0f}')
print(f'\nfor reference, score_max = {score_max:.0f}')

# spot check predictions vs actuals
preds = model.predict(x_val[:10], verbose=0).flatten()
actuals = y_val[:10]
print(f'\nspot check (normalized):')
for i in range(10):
    print(f'  predicted: {preds[i]:.4f}  actual: {actuals[i]:.4f}  '
          f'error: {abs(preds[i]-actuals[i])*score_max:.0f}')

validation loss (mse): 0.022262
validation mae (normalized): 0.1153
validation mae (score units): 7162

for reference, score_max = 62144

spot check (normalized):
  predicted: 0.2289  actual: 0.3079  error: 4905
  predicted: 0.0438  actual: 0.0196  error: 1506
  predicted: 0.2902  actual: 0.3755  error: 5300
  predicted: 0.2237  actual: 0.0938  error: 8070
  predicted: 0.1528  actual: 0.1167  error: 2244
  predicted: 0.2307  actual: 0.0951  error: 8429
  predicted: 0.1500  actual: 0.0951  error: 3415
  predicted: 0.1561  actual: 0.5147  error: 22288
  predicted: 0.0377  actual: 0.0549  error: 1070
  predicted: 0.1221  actual: 0.1386  error: 1023


## save keras model

In [6]:
model.save('board_evaluator.keras')
print('saved board_evaluator.keras')

# also save score_max so js can denormalize predictions
np.save('score_max.npy', np.array([score_max], dtype=np.float32))
print(f'saved score_max.npy (value: {score_max:.0f})')

saved board_evaluator.keras
saved score_max.npy (value: 62144)


## export to tensorflow.js

In [7]:
!pip install tensorflowjs -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 73.9 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
build 1.6.1 requires packaging>=24.0, but you have packaging 20.9 which is incompatible.
lightning-utilities 0.15.3 requires packaging>=22, but you have packaging 20.9 which is incompatible.
pytesseract 0.3.13 requires packaging>=21.3, but you have packaging 20.9 which is incompatib

In [ ]:
import os
import json
import numpy as np

# export model to tensorflow.js format
def export_to_tfjs(model, output_dir='tfjs_model'):
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. attempt conversion via tf_keras and tensorflowjs if available and compatible
    try:
        import tf_keras
        import tensorflowjs as tfjs
        legacy_model = tf_keras.Sequential([
            tf_keras.layers.InputLayer(input_shape=(16,)),
            tf_keras.layers.Dense(128, activation='relu', name=model.layers[0].name),
            tf_keras.layers.Dense(64, activation='relu', name=model.layers[1].name),
            tf_keras.layers.Dense(1, activation='linear', name=model.layers[2].name)
        ])
        legacy_model.set_weights(model.get_weights())
        tfjs.converters.save_keras_model(legacy_model, output_dir)
        return
    except Exception:
        pass
    
    # 2. direct export fallback: write canonical model.json and weights binary shard
    weight_entries = []
    weight_bytes = bytearray()
    
    for layer in model.layers:
        weights = layer.get_weights()
        if not weights:
            continue
        kernel, bias = weights[0], weights[1]
        
        k_data = kernel.astype('<f4')
        weight_entries.append({
            "name": f"{layer.name}/kernel",
            "shape": list(k_data.shape),
            "dtype": "float32"
        })
        weight_bytes.extend(k_data.tobytes())
        
        b_data = bias.astype('<f4')
        weight_entries.append({
            "name": f"{layer.name}/bias",
            "shape": list(b_data.shape),
            "dtype": "float32"
        })
        weight_bytes.extend(b_data.tobytes())
        
    bin_filename = "group1-shard1of1.bin"
    with open(os.path.join(output_dir, bin_filename), 'wb') as f:
        f.write(weight_bytes)
        
    layers_config = [
        {
            "class_name": "InputLayer",
            "config": {
                "batch_input_shape": [None, 16],
                "dtype": "float32",
                "sparse": False,
                "name": f"{model.layers[0].name}_input"
            }
        }
    ]
    
    for layer in model.layers:
        act = getattr(layer.activation, '__name__', str(layer.activation))
        layers_config.append({
            "class_name": "Dense",
            "config": {
                "name": layer.name,
                "trainable": True,
                "dtype": "float32",
                "units": layer.units,
                "activation": act,
                "use_bias": layer.use_bias,
                "kernel_initializer": {"class_name": "GlorotUniform", "config": {"seed": None}},
                "bias_initializer": {"class_name": "Zeros", "config": {}},
                "kernel_regularizer": None,
                "bias_regularizer": None,
                "activity_regularizer": None,
                "kernel_constraint": None,
                "bias_constraint": None
            }
        })
        
    model_json = {
        "format": "layers-model",
        "generatedBy": "keras v3",
        "convertedBy": "TensorFlow.js Converter",
        "modelTopology": {
            "class_name": "Sequential",
            "config": {
                "name": model.name,
                "layers": layers_config
            },
            "keras_version": "2.15.0",
            "backend": "tensorflow"
        },
        "weightsManifest": [
            {
                "paths": [bin_filename],
                "weights": weight_entries
            }
        ]
    }
    
    with open(os.path.join(output_dir, 'model.json'), 'w', encoding='utf-8') as f:
        json.dump(model_json, f, indent=2)

export_to_tfjs(model, 'tfjs_model')
print('exported to tfjs_model/')

for f in os.listdir('tfjs_model'):
    size = os.path.getsize(os.path.join('tfjs_model', f))
    print(f'  {f} ({size} bytes)')

exported to tfjs_model/
  model.json (3781 bytes)
  group1-shard1of1.bin (41988 bytes)


/usr/local/lib/python3.13/dist-packages/tensorflowjs/read_weights.py:28: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  np.uint8, np.uint16, np.object, np.bool]


## package output

In [9]:
import shutil, os

os.makedirs('output', exist_ok=True)

# copy tfjs model files
for f in os.listdir('tfjs_model'):
    shutil.copy(os.path.join('tfjs_model', f), os.path.join('output', f))

# copy score_max
shutil.copy('score_max.npy', 'output/score_max.npy')

shutil.make_archive('02_model_training_output', 'zip', 'output')
print('done. download 02_model_training_output.zip')
print('\ncontents:')
for f in os.listdir('output'):
    print(f'  {f}')

done. download 02_model_training_output.zip

contents:
  score_max.npy
  model.json
  group1-shard1of1.bin
